[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/05_Visualising_Networks/04_nxpandas.ipynb)

# Multi-omics Data Science

# Pandas and NetworkX practical (co-abundance network analysis of *Klebsiella pneumoniae* sepsis)

Using the course dataset we will build a **protein co-abundance network** from serum proteomics of septic patients, and then export it to **Cytoscape** for publication-quality figures.

Steps:
- Read the data using pandas
- Run a short exploratory analysis and filter the data
- Compute a protein-protein correlation matrix
- Turn the matrix into an edge list
- Build a network from the edge list
- Visualise the network
- Find the central nodes
- Attach node attributes and export everything to Cytoscape

## Loading external data

We use the course dataset, which comes from *Serum proteomic profiling of patients with carbapenem-resistant and carbapenem-susceptible* Klebsiella pneumoniae *sepsis* (He J et al., Front Immunol 2026;17:1818068).

Three files are relevant for this notebook, all of them in the course repository:

| File | Content |
| --- | --- |
| `proteomics/data/protein_groups_matrix.tsv` | MaxLFQ protein intensities: 4 annotation columns (`protein_group`, `protein_names`, `genes`, `description`), then 45 patient samples (`Con1`-`Con15`, `KP1`-`KP15`, `CRKP1`-`CRKP15`) and 3 pooled quality-control runs (`QC_pool1`-`QC_pool3`) |
| `metadata/sample_metadata.tsv` | One row per sample: `sample_id`, `group`, `group_order`, `age`, `sex` and clinical laboratory values |
| `proteomics/data/published_deps.tsv` | Differentially expressed proteins reported in the paper, per `comparison` (`CRKP_vs_KP`, `CRKP_vs_Con`, `KP_vs_Con`, `CRKP_vs_KP_vs_Con`), with `log2_fold_change` and `p_value` |

**Study background**

Serum was collected on the day of diagnosis (day 0) from 45 patients with sepsis, 15 per group:

- `Con` - sepsis with negative blood cultures,
- `CSKP` - sepsis caused by carbapenem-**susceptible** *Klebsiella pneumoniae* (the sample identifiers of this group start with `KP`),
- `CRKP` - sepsis caused by carbapenem-**resistant** *K. pneumoniae*.

The samples were measured by diaPASEF on a timsTOF Pro mass spectrometer and processed with DIA-NN, giving MaxLFQ intensities for 1458 protein groups with about 27% missing values.

The clinical question is a practical one: **can host molecules in serum tell a resistant infection apart from a susceptible one on day 0**, before the culture and antibiogram results come back two or three days later? Antibiotic choice on day 0 changes outcomes, so a host-response signature that is available immediately would be useful.

In this notebook we do not test that question directly. Instead we ask a structural question about the same data: **which proteins move together across these 45 patients**, and what does the resulting network look like?

### Reading data with pandas

**Comma separated / tab separated files (.csv, .tsv)**

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"


def data_path(relative_path):
    """Locate a course data file.

    If the file exists in a local clone of the course repository (searching the
    current folder and its parents) we use that copy; otherwise we fall back to
    reading it straight from GitHub, which is what happens on Google Colab.
    """
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / relative_path
        if candidate.exists():
            return str(candidate)
    return f"{BASE_URL}/{relative_path}"


PROTEIN_MATRIX = data_path("proteomics/data/protein_groups_matrix.tsv")
SAMPLE_METADATA = data_path("metadata/sample_metadata.tsv")
PUBLISHED_DEPS = data_path("proteomics/data/published_deps.tsv")

print("protein matrix :", PROTEIN_MATRIX)
print("sample metadata:", SAMPLE_METADATA)
print("published DEPs :", PUBLISHED_DEPS)

In [ ]:
# pandas reads a tab separated file with read_csv and sep="\t"
deps = pd.read_csv(PUBLISHED_DEPS, sep="\t")
deps.head()

## Exploratory analysis

**Look at the first 10 rows**

In [ ]:
deps.head(10)

**Look at the last 10 rows**

In [ ]:
deps.tail(10)

**Show summary statistics for the columns**

In [ ]:
deps.describe()

**Show the columns**

In [ ]:
print(deps.columns.tolist())
print()
print(deps["comparison"].value_counts())

**Drop every column except: `protein_group`, `genes`, `log2_fold_change`, `p_value`**

Use the **drop** method https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html

In [ ]:
# keep only the comparison we care about most: resistant vs susceptible K. pneumoniae
crkp_vs_kp = deps[deps["comparison"] == "CRKP_vs_KP"]

# drop the columns we do not need (axis=1 means "columns", not "rows")
crkp_vs_kp = crkp_vs_kp.drop(
    ["comparison", "mean_case", "mean_control", "fold_change"], axis=1
)

print(crkp_vs_kp.shape)
crkp_vs_kp.sort_values("log2_fold_change").head()

## Protein-protein correlation matrix

**Using the protein intensity matrix of the course dataset, compute a correlation matrix with the pandas [corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) method**

`corr` correlates the **columns** of a data frame. Our matrix has one row per protein group and one column per sample, so to obtain a **protein-protein** correlation matrix we set `protein_group` as the index and **transpose** the matrix before calling `corr`.

Two things to take care of first:

1. Separate the 4 annotation columns and the 3 `QC_pool*` columns from the 45 patient columns. The QC pools are the same pooled sample injected repeatedly to monitor instrument drift; they are not biological replicates and must not enter a correlation across patients.
2. Work on `log2` intensities. MaxLFQ intensities span several orders of magnitude, and on the log scale the distribution is roughly symmetric, which is what a Pearson correlation expects.

In [ ]:
proteins = pd.read_csv(PROTEIN_MATRIX, sep="\t")

annotation_columns = ["protein_group", "protein_names", "genes", "description"]
qc_columns = [c for c in proteins.columns if c.startswith("QC_pool")]
patient_columns = [
    c for c in proteins.columns if c not in annotation_columns + qc_columns
]

# annotation table: one row per protein group, indexed by protein_group
annotation = proteins[annotation_columns].set_index("protein_group")

# numeric matrix: proteins (rows) x patient samples (columns), on the log2 scale
intensities = proteins.set_index("protein_group")[patient_columns]
log_intensities = np.log2(intensities)

print(f"{proteins.shape[0]} protein groups")
print(f"{len(patient_columns)} patient samples, e.g. {patient_columns[:3]} ... {patient_columns[-3:]}")
print(f"{len(qc_columns)} QC pools kept aside: {qc_columns}")

# sanity check against the metadata: the 45 columns should be exactly the 45 samples
metadata = pd.read_csv(SAMPLE_METADATA, sep="\t").set_index("sample_id")
assert set(patient_columns) == set(metadata.index), "columns and metadata disagree"
print()
print(metadata["group"].value_counts().sort_index())

log_intensities.iloc[:5, :5]

In [ ]:
print("matrix shape (proteins x samples):", log_intensities.shape)
print(f"missing values: {100 * log_intensities.isna().mean().mean():.1f}%")

# in how many of the 45 patients was each protein quantified?
observed_per_protein = log_intensities.notna().sum(axis=1)

values = log_intensities.to_numpy().ravel()
values = values[~np.isnan(values)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed_per_protein, bins=45, color="#4878A8")
axes[0].set_xlabel("number of samples with a value")
axes[0].set_ylabel("number of protein groups")
axes[0].set_title("Completeness per protein")
axes[1].hist(values, bins=60, color="#A85878")
axes[1].set_xlabel("log2 MaxLFQ intensity")
axes[1].set_ylabel("number of measurements")
axes[1].set_title("Intensity distribution")
fig.tight_layout()
plt.show()

print(f"{(observed_per_protein == len(patient_columns)).sum()} proteins are complete in all {len(patient_columns)} samples")

In [ ]:
# Compute the correlation matrix.
#
# Keeping every protein would give a 1458 x 1458 matrix (over a million pairs),
# most of it driven by missing values. We therefore keep
#   (a) only proteins measured in ALL 45 patients, and
#   (b) among those, the 150 most variable ones - a protein that barely changes
#       between patients cannot tell us anything about co-regulation.
complete = log_intensities.dropna(axis=0)

variability = complete.std(axis=1).sort_values(ascending=False)
N_PROTEINS = 150
most_variable = complete.loc[variability.index[:N_PROTEINS]]

# transpose: samples become the rows, proteins the columns, so corr() gives us
# a protein x protein matrix of Pearson correlation coefficients
corr_matrix = most_variable.T.corr()
corr_matrix = corr_matrix.rename_axis(index="source", columns="target")

print("proteins measured in every sample:", complete.shape[0])
print("correlation matrix shape:", corr_matrix.shape)
corr_matrix.iloc[:5, :5].round(2)

## Turning the matrix into an edge list

Using the pandas **stack** method, convert the matrix into an edge list: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html

A correlation matrix is symmetric and its diagonal is 1 (every protein correlates perfectly with itself). We therefore keep only the **upper triangle**, so that each protein pair appears exactly once and no self-loops are created.

In [ ]:
# keep the upper triangle only (k=1 excludes the diagonal)
upper_triangle = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)

# turn the data frame corr_matrix into an edge list
edges = corr_matrix.where(upper_triangle).stack().reset_index()
edges = edges.dropna()

print(f"{len(edges)} protein pairs")
edges.head()

**Rename the columns to: source, target, weight**

In [ ]:
edges.columns = ["source", "target", "weight"]
edges.head()

**Keep only the pairs whose correlation is strong: filter out every correlation below 0.7 in absolute value**

To take the absolute value you can import **numpy** and use the **absolute** method https://numpy.org/doc/stable/reference/generated/numpy.absolute.html, or the `.abs()` method of a pandas Series.

Note that a negative correlation is just as informative as a positive one: two proteins that consistently move in opposite directions are also linked. That is why we threshold on the absolute value and keep the sign as an edge attribute.

**How strong is strong?** A threshold is a claim about evidence, and the same number means different things with different sample sizes. With n = 45 patients, |r| = 0.7 is far beyond what noise produces; with only n = 15 samples, |r| = 0.7 happens by chance often enough that a network built on it would be mostly artefact. Choose the threshold with the sample size in mind, and be explicit about it in a paper.

In [ ]:
THRESHOLD = 0.7

strong_edges = edges[np.absolute(edges["weight"]) >= THRESHOLD].copy()

print(f"{len(edges)} pairs -> {len(strong_edges)} pairs with |r| >= {THRESHOLD}")
print(f"  positive correlations: {(strong_edges['weight'] > 0).sum()}")
print(f"  negative correlations: {(strong_edges['weight'] < 0).sum()}")
strong_edges.sort_values("weight").head()

## Building the co-abundance network

Remember to keep the "weight" attribute!

**What does an edge mean here?** An edge between two proteins means that their abundances rise and fall together across these 45 patients. That is a hypothesis about shared regulation: the two proteins may be controlled by the same upstream signal, released from the same tissue, or cleared by the same route. It is **not** evidence of a physical interaction, and it is not evidence of a common pathway either. Correlation networks and interaction networks look identical on screen and mean very different things.

In [ ]:
import networkx as nx

In [ ]:
# nx.from_pandas_edgelist builds the graph directly from the data frame;
# edge_attr="weight" carries the correlation coefficient onto each edge
G = nx.from_pandas_edgelist(
    strong_edges, source="source", target="target", edge_attr="weight"
)

# nodes are UniProt protein group identifiers; gene symbols are easier to read,
# so we attach them as a node attribute (falling back to the identifier when a
# protein group has no gene symbol)
gene_of = annotation["genes"].fillna(annotation.index.to_series()).to_dict()
nx.set_node_attributes(G, {n: gene_of.get(n, n) for n in G.nodes()}, name="gene")

labels = nx.get_node_attributes(G, "gene")

print(G)
print("example nodes:", [labels[n] for n in list(G.nodes())[:8]])

**Get the number of nodes**

In [ ]:
print(G.number_of_nodes())

**Get the number of edges**

In [ ]:
print(G.number_of_edges())

**Get the degree of the nodes**

In [ ]:
degree_table = (
    pd.Series(dict(G.degree()), name="degree")
    .rename_axis("protein_group")
    .to_frame()
)
degree_table["gene"] = [gene_of.get(p, p) for p in degree_table.index]
degree_table = degree_table.sort_values("degree", ascending=False)

print("mean degree:", round(degree_table["degree"].mean(), 2))
print("connected components:", nx.number_connected_components(G))
degree_table.head(10)

## Visualising the network

**Use several layouts to visualise the network**

A layout is only a way of placing the nodes on the page: it carries no biological meaning. The same network drawn with two different layouts can look like two different results, so never read a claim into the positions alone.

In [ ]:
pos_spring = nx.spring_layout(G, seed=42)
pos_circular = nx.circular_layout(G)
# some layouts interpret "weight" as a distance and cannot handle negative
# values, so we tell kamada_kawai to ignore the correlation sign
pos_kk = nx.kamada_kawai_layout(G, weight=None)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, (name, pos) in zip(
    axes,
    [("spring", pos_spring), ("circular", pos_circular), ("kamada-kawai", pos_kk)],
):
    nx.draw_networkx(
        G,
        pos=pos,
        ax=ax,
        with_labels=False,
        node_size=60,
        node_color="#4878A8",
        edge_color="#CCCCCC",
        width=0.6,
    )
    ax.set_title(f"{name} layout")
    ax.axis("off")
fig.tight_layout()
plt.show()

**Show the node names**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(G, pos=pos_spring, ax=ax, edge_color="#DDDDDD", width=0.6)
nx.draw_networkx_nodes(G, pos=pos_spring, ax=ax, node_size=120, node_color="#4878A8")
# labels comes from the "gene" node attribute we set above
nx.draw_networkx_labels(G, pos=pos_spring, labels=labels, ax=ax, font_size=7)
ax.set_title(f"Serum protein co-abundance network (|r| >= {THRESHOLD}, n = 45 patients)")
ax.axis("off")
fig.tight_layout()
plt.show()

**Change the colour of the nodes and colour the edges as well**

Colour should encode something. Here we colour the **nodes** by their degree and the **edges** by the sign of the correlation: red for proteins that move together, blue for proteins that move in opposite directions.

See this page for an example of how to colour a network: https://networkx.org/documentation/stable/auto_examples/drawing/plot_labels_and_colors.html

In [ ]:
node_degrees = [G.degree(n) for n in G.nodes()]
edge_colors = [
    "#C0392B" if G.edges[e]["weight"] > 0 else "#2471A3" for e in G.edges()
]
edge_widths = [2.5 * abs(G.edges[e]["weight"]) for e in G.edges()]

fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(
    G, pos=pos_spring, ax=ax, edge_color=edge_colors, width=edge_widths, alpha=0.6
)
nodes = nx.draw_networkx_nodes(
    G,
    pos=pos_spring,
    ax=ax,
    node_size=[40 + 25 * d for d in node_degrees],
    node_color=node_degrees,
    cmap=plt.cm.viridis,
)
nx.draw_networkx_labels(G, pos=pos_spring, labels=labels, ax=ax, font_size=7)
fig.colorbar(nodes, ax=ax, label="degree", shrink=0.6)
ax.set_title("Node colour and size = degree; edge colour = sign of the correlation")
ax.axis("off")
fig.tight_layout()
plt.show()

## Central nodes

Use some of the functions NetworkX provides to compute centrality (https://networkx.org/documentation/stable/reference/algorithms/centrality.html).
Centrality is a way of ranking the nodes of a network by their number of connections, their position in the topology, or the flow of information through them (https://en.wikipedia.org/wiki/Centrality).

Three measures are enough for most purposes:

- **degree centrality** - how many neighbours a node has, divided by the maximum possible;
- **betweenness centrality** - how often a node lies on the shortest path between two other nodes; high betweenness marks a bridge between parts of the network;
- **closeness centrality** - how short the paths from a node to all others are on average.

In [ ]:
centrality = pd.DataFrame(
    {
        "degree": nx.degree_centrality(G),
        "betweenness": nx.betweenness_centrality(G),
        "closeness": nx.closeness_centrality(G),
    }
).rename_axis("protein_group")
centrality["gene"] = [gene_of.get(p, p) for p in centrality.index]

print("Top 10 by betweenness centrality")
centrality.sort_values("betweenness", ascending=False).head(10).round(3)

**What centrality does and does not mean in a correlation network**

In a network of physical interactions, a node with high betweenness really can be a bottleneck: information, or a signal, has to pass through it. A co-abundance network has no flow, so the interpretation is much weaker.

Read the numbers with these caveats in mind:

- **Nothing travels along these edges.** A "shortest path" between two proteins is a property of our correlation matrix, not a route taken by molecules. Betweenness therefore describes the shape of our table, not a mechanism.
- **Degree depends on the threshold.** Move the threshold from 0.7 to 0.6 and the hubs change. Any conclusion that survives only at one threshold is not a conclusion.
- **Abundant, co-regulated protein families inflate degree.** Serum is dominated by a few families (immunoglobulins, apolipoproteins, complement, acute-phase proteins) whose members are quantified from overlapping peptides and are genuinely co-regulated. They form dense cliques and their members will top any centrality ranking, which is a fact about serum biology and about protein inference, not a discovery.
- **A single upstream driver produces a hub-shaped cluster.** If one inflammatory signal moves twenty proteins, all twenty correlate with each other. The most central of them is not the cause; it is just the best-measured member of the group.
- **We used a subset.** We kept 150 complete, highly variable proteins. Centrality was computed on that subset, and a protein absent from the subset cannot be central by construction.

Used properly, centrality is a way of **prioritising candidates for follow-up**, not evidence in itself.

## Node attributes worth carrying into Cytoscape

A network on its own is only topology. What makes a Cytoscape figure informative is what we attach to the nodes. For each protein in our network we collect:

- `gene` - the gene symbol, to use as the node label;
- `degree` and `betweenness` - the network measures we just computed;
- `is_dep_CRKP_vs_KP` - 1 if the protein was reported as differentially expressed between resistant and susceptible *K. pneumoniae* sepsis in the paper, 0 otherwise;
- `log2fc_CRKP_vs_KP` and `log2fc_CRKP_vs_Con` - the published log2 fold changes, to colour the nodes with.

We get the published results by joining `published_deps.tsv` on `protein_group`. Proteins that were not reported for a comparison have no fold change; we store `0.0` for them and keep the flag alongside, so that "not reported" is never mistaken for "no change".

In [ ]:
deps_all = pd.read_csv(PUBLISHED_DEPS, sep="\t")

dep_crkp_vs_kp = deps_all[deps_all["comparison"] == "CRKP_vs_KP"].set_index("protein_group")
dep_crkp_vs_con = deps_all[deps_all["comparison"] == "CRKP_vs_Con"].set_index("protein_group")

node_table = pd.DataFrame(index=pd.Index(sorted(G.nodes()), name="protein_group"))
node_table["gene"] = [gene_of.get(p, p) for p in node_table.index]
node_table["description"] = annotation["description"].reindex(node_table.index)
node_table["degree"] = [G.degree(p) for p in node_table.index]
node_table["betweenness"] = centrality["betweenness"].reindex(node_table.index).round(4)

# stored as 1/0 rather than True/False: both GraphML and Cytoscape handle
# integers unambiguously, and a discrete mapping on 1/0 is easy to set up
node_table["is_dep_CRKP_vs_KP"] = node_table.index.isin(dep_crkp_vs_kp.index).astype(int)
node_table["log2fc_CRKP_vs_KP"] = (
    dep_crkp_vs_kp["log2_fold_change"].reindex(node_table.index).fillna(0.0).round(3)
)
node_table["log2fc_CRKP_vs_Con"] = (
    dep_crkp_vs_con["log2_fold_change"].reindex(node_table.index).fillna(0.0).round(3)
)

# copy the attributes onto the graph itself, so that they are written to GraphML
for column in node_table.columns:
    nx.set_node_attributes(G, node_table[column].to_dict(), name=column)

print(f"{node_table['is_dep_CRKP_vs_KP'].sum()} of {len(node_table)} network proteins are "
      "published CRKP vs KP differential proteins")
node_table.sort_values("degree", ascending=False).head(10)

## Exporting the network to Cytoscape

There are two file routes into Cytoscape, and it is worth writing both:

1. **GraphML** (`.graphml`) - one file that contains the topology, the edge weights and all node attributes. NetworkX writes it with `nx.write_graphml`. Cytoscape also reads GML (`nx.write_gml`), but GML handles attributes less cleanly.
2. **An edge table + a node attribute table** as two CSV files. This is more work to load, but it is the route that always works, it opens in Excel or R, and it is what you would deposit as supplementary material.

In [ ]:
OUT_DIR = Path("cytoscape_export")
OUT_DIR.mkdir(exist_ok=True)

# 1. GraphML: topology + edge weights + every node attribute in a single file
graphml_file = OUT_DIR / "coabundance_network.graphml"
nx.write_graphml(G, graphml_file)

# 2a. edge table
edge_table = strong_edges.rename(columns={"weight": "pearson_r"}).copy()
edge_table["source_gene"] = [gene_of.get(p, p) for p in edge_table["source"]]
edge_table["target_gene"] = [gene_of.get(p, p) for p in edge_table["target"]]
edge_table["abs_r"] = edge_table["pearson_r"].abs().round(3)
edge_table["sign"] = np.where(edge_table["pearson_r"] > 0, "positive", "negative")
edge_table["pearson_r"] = edge_table["pearson_r"].round(3)
edge_table = edge_table[
    ["source", "target", "source_gene", "target_gene", "pearson_r", "abs_r", "sign"]
]
edge_file = OUT_DIR / "coabundance_edges.csv"
edge_table.to_csv(edge_file, index=False)

# 2b. node attribute table
node_file = OUT_DIR / "coabundance_nodes.csv"
node_table.to_csv(node_file)

for f in [graphml_file, edge_file, node_file]:
    print(f"{f}  ({f.stat().st_size / 1024:.1f} kB)")

edge_table.head()

### How to load these files in Cytoscape

Start Cytoscape (3.10 or newer) and then, depending on which file you want to use:

**Route 1 - the GraphML file (simplest, one step)**

1. `File -> Import -> Network from File...` and choose `coabundance_network.graphml`.
2. The network appears with every node attribute already in the node table. Nothing else to import.

**Route 2 - the two CSV files**

1. `File -> Import -> Network from File...` and choose `coabundance_edges.csv`.
   In the import dialog set the column meanings: `source` = **Source Node**, `target` = **Target Node**, and `pearson_r`, `abs_r`, `sign` = **Edge Attribute**. Click OK.
2. `File -> Import -> Table from File...` and choose `coabundance_nodes.csv`.
   Set **Key Column for Network** to `shared name` and the key column of the file to `protein_group`, so the rows are matched to the nodes that already exist. The remaining columns are imported as node attributes.

**Then style the network** in the `Style` panel (left-hand side, `Style` tab):

| Visual property | Column | Mapping type | Suggested setting |
| --- | --- | --- | --- |
| **Label** | `gene` | Passthrough Mapping | gene symbols as node labels |
| **Fill Colour** | `log2fc_CRKP_vs_KP` | Continuous Mapping | diverging palette, blue - white - red, centred on 0 |
| **Size** | `degree` | Continuous Mapping | e.g. 20 for the lowest degree, 70 for the highest |
| **Border Width** | `is_dep_CRKP_vs_KP` | Discrete Mapping | 1 -> thick border (published differential protein), 0 -> thin |
| **Edge Stroke Colour** | `sign` | Discrete Mapping | red = positive, blue = negative |
| **Edge Width** | `abs_r` | Continuous Mapping | thicker for stronger correlations |

A few practical points:

- A fold change is a signed quantity, so it needs a **diverging** palette centred on zero. A sequential palette (light to dark) hides the direction of the change, which is usually the whole point of the figure.
- Choose the layout in `Layout -> Prefuse Force Directed Layout` (or `yFiles Organic`, if installed). If you want the strong correlations to pull nodes closer together, set the edge weight column to `abs_r` in the layout settings.
- `Layout -> Apply Preferred Layout` after importing; Cytoscape's default grid layout tells you nothing.
- Export the figure with `File -> Export -> Network to Image...` (PDF or SVG for a paper, not PNG).

### Driving Cytoscape from Python

If Cytoscape is running on the same machine, you can skip the files entirely and control it from a notebook with **py4cytoscape**, which talks to the CyREST interface on `localhost:1234`:

```python
# pip install py4cytoscape   (not installed in this course environment)
import py4cytoscape as p4c

p4c.cytoscape_ping()                      # check that Cytoscape is listening
p4c.create_network_from_networkx(G, title="Serum co-abundance", collection="Course")
p4c.set_node_label_mapping("gene")
p4c.set_node_color_mapping(
    "log2fc_CRKP_vs_KP", [-2, 0, 2], ["#2471A3", "#FFFFFF", "#C0392B"]
)
p4c.set_node_size_mapping("degree", [1, 30], [20, 70])
p4c.layout_network("force-directed")
```

This is convenient for a reproducible pipeline, but it only works while a local Cytoscape is open, and it cannot run on Colab. **The file route always works**, so export the GraphML and the CSVs even when you use py4cytoscape.

### Where this leaves us

We turned a 1458 x 45 intensity matrix into a network of correlated serum proteins, ranked its nodes, and annotated them with the published differential proteins. Remember what that network is: a picture of which proteins move together in these 45 patients. It suggests co-regulation, a shared tissue of origin or a shared upstream driver, and it gives you a shortlist to test - it does not show physical interactions, and it does not by itself answer whether serum can distinguish resistant from susceptible infection on day 0.